# CH2 — 프롬프트 엔지니어링: Agent에게 일을 시키는 법

CH1에서 Agent가 **다음 행동을 스스로 결정**하는 걸 봤습니다.
그런데 그 결정은 어디서 나왔을까요? → 전부 **프롬프트** 안에 있었습니다.

> ChatGPT 시대의 프롬프트 = 질문
> **Agent 시대의 프롬프트 = 업무 지시서 + 판단 기준표**

## 오늘 볼 3가지 실험
1. **나쁜 지시 vs 좋은 지시** — 같은 자료, 지시만 바꾸면?
2. **출력 형식 고정** — 코드가 읽을 수 있게 JSON으로
3. **Agent 결정 프롬프트 해부** — 완료 기준 한 줄 바꾸면 행동이 바뀐다

## 0. 환경 준비 (CH1과 같은 데이터 사용)

In [ ]:
from pathlib import Path
import os, json, re
from dotenv import load_dotenv
from openai import OpenAI

# data 폴더가 보일 때까지 위로 올라가며 프로젝트 루트 찾기
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
client = OpenAI()

brief = (PROJECT_ROOT / "data" / "company_brief.md").read_text(encoding="utf-8")
db = json.loads((PROJECT_ROOT / "data" / "competitor_db.json").read_text(encoding="utf-8"))

def ask(prompt):
    return client.responses.create(model=MODEL, input=prompt).output_text

print("ROOT:", PROJECT_ROOT)
print("경쟁사:", list(db.keys()))

---
# 실험 1. 나쁜 지시 vs 좋은 지시

**같은 자료**(MeetNote AI 전체 데이터)를 줍니다. 바뀌는 건 지시문뿐입니다.

In [ ]:
data = json.dumps(db["MeetNote AI"], ensure_ascii=False, indent=2)

bad_prompt = f"""
경쟁사 분석해줘.

{data}
"""

bad = ask(bad_prompt)
print(bad)

### 관찰
- 그럴듯하지만 **우리 회사와 무슨 상관인지**가 빠져 있죠?
- 길이·형식도 매번 다릅니다 → 다음 단계(코드/사람)가 쓰기 어렵습니다.

이제 **업무 지시서 6요소**를 넣어봅니다.

| 요소 | 질문 |
|---|---|
| 역할 | 누구로서 일하나? |
| 목표 | 이 결과물로 무슨 결정을 하나? |
| 맥락 | 우리 상황은? (브리프) |
| 제약 | 하면 안 되는 것은? (추측 금지 등) |
| 출력 형식 | 어떤 모양으로? |
| 완료 기준 | 언제 "다 했다"인가? |

In [ ]:
good_prompt = f"""
[역할]
너는 B2B SaaS 사업전략 담당자다.

[목표]
우리 팀이 'MeetNote AI와 정면승부할지, 피해갈지'를 결정할 수 있게 돕는다.

[맥락 - 우리 회사 브리프]
{brief}

[자료]
{data}

[제약]
- 자료에 없는 내용은 추측하지 말고 "정보 부족"이라고 쓴다.
- 모든 판단 뒤에 (근거: 필드명)을 붙인다.

[출력 형식]
1. 한 줄 결론
2. 우리 가설 4개 각각에 대한 검증 결과 (표: 가설 | 경쟁사 현황 | 우리 기회)
3. 차별화 포인트 2개

[완료 기준]
브리프의 핵심 가설 4개가 모두 표에 들어가 있으면 끝.
"""

good = ask(good_prompt)
print(good)

### 💬 질문 던지기
- 두 결과 중 **내일 회의에 바로 들고 갈 수 있는 건?**
- 6요소 중 하나만 지운다면, 어떤 걸 지웠을 때 가장 망가질까? → 직접 지워보세요!

> AI는 똑똑한 **신입사원**입니다. 신입한테 "분석해줘"만 던지면 안 되잖아요?

---
# 실험 2. 출력 형식 고정 — 사람 말고 **코드**가 읽는 출력

Agent에서는 LLM 출력을 **코드가 받아서 다음 행동을 실행**합니다.
그래서 "예쁜 글"보다 **파싱 가능한 형식**이 더 중요합니다.

In [ ]:
loose = ask("""
MeetNote AI의 다음 조사 항목을 추천해줘.
지금 가진 정보: overview, pricing
""")
print(loose)

try:
    json.loads(loose)
except Exception as e:
    print("\n❌ 코드가 못 읽음:", type(e).__name__)

In [ ]:
def extract_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    return json.loads(m.group(0))

strict = ask("""
MeetNote AI의 다음 조사 항목을 하나만 골라라.
지금 가진 정보: overview, pricing
선택 가능: features, reviews, security, target

아래 JSON 형식으로만 답하라. 설명 문장 금지.
{"field": "선택한 항목", "reason": "한 문장"}
""")
print(strict)

decision = extract_json(strict)
print("\n✅ 코드가 읽음 →", decision["field"])

### 포인트
- `"JSON으로만 답하라"` + **예시 스키마**를 보여주는 게 핵심
- 이 한 줄 덕분에 CH1 Agent 루프가 `decision["action"]`으로 돌아갈 수 있었습니다
- (PART 3에서 Tool Calling / Structured Output으로 더 안전하게 바꿉니다)

---
# 실험 3. Agent 결정 프롬프트 해부

CH1의 `agent_decide()` 프롬프트를 축소판으로 가져왔습니다.
**완료 기준**을 바꾸면 Agent의 행동이 어떻게 바뀌는지 봅니다.

In [ ]:
state = {
    "MeetNote AI": {"overview": "...", "pricing": "...", "features": "..."},
}

def decide(state, done_criteria):
    prompt = f"""
너는 경쟁사 리서치 Agent다.                         # ← 역할

목표: 우리 회사 브리프에 맞는 경쟁사 분석 근거를 모은다.   # ← 목표

[현재까지 수집된 정보]                               # ← 상태(State)
{json.dumps(state, ensure_ascii=False)}

[사용 가능한 field]                                 # ← 쓸 수 있는 도구
overview, pricing, features, reviews, security, target

[완료 기준]                                         # ← 멈출 조건
{done_criteria}

다음 행동을 하나만 골라 JSON으로만 답하라.             # ← 출력 형식
{{"action": "search", "field": "...", "reason": "..."}}
또는
{{"action": "finish", "reason": "..."}}
"""
    return extract_json(ask(prompt))

print("A) 기준 느슨:", decide(state, "overview, pricing, features가 있으면 완료"))
print("B) 기준 엄격:", decide(state, "overview, pricing, features, reviews, security, target 모두 있어야 완료"))

### 관찰
- **A**: 이미 다 있으니 `finish`
- **B**: 부족한 `reviews`나 `security`를 찾으러 `search`

코드는 한 줄도 안 바꿨습니다. **판단 기준(프롬프트)만 바꿨는데 행동이 바뀌었습니다.**

> Agent 설계 = 행동을 코딩하는 게 아니라, **판단 기준을 글로 쓰는 것**
> → "기획은 글로 하는 코딩"

---
# 🧪 셀프 실습 (5분)

1. 실험 1의 `good_prompt`에서 `[제약]` 블록을 지우고 다시 실행 → 무엇이 달라졌나?
2. 실험 3에 완료 기준 C를 직접 써보세요.
   예: `"보안 정보가 '정보 부족'이면 그 회사는 제외하고 완료"`
3. 결과를 채팅창에 공유!

# 오늘의 한 문장
> ## Agent의 행동은 코드가 아니라 **프롬프트 속 판단 기준**에서 나온다.